# Intermezzo · L'attrezzeria

La nostra rete scritta in PyTorch, e il gradiente come trasformazione in JAX. Servono `torch` e `jax`.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def sigmoide(z):
    return 1 / (1 + math.exp(-z))

## PyTorch: la rete 3 → 2 → 1

Il kata chiede di provare tre learning rate partendo sempre dagli stessi pesi: per questo il seme si fissa prima di creare la rete.

In [2]:
import torch
import torch.nn as nn

def allena(lr, passi=100, seme=0):
    torch.manual_seed(seme)
    rete = nn.Sequential(nn.Linear(3, 2), nn.ReLU(), nn.Linear(2, 1))
    x = torch.tensor([[0.8, 1.0, 1.0]])
    y = torch.tensor([[0.0]])
    errore = nn.BCEWithLogitsLoss()
    ottimizzatore = torch.optim.SGD(rete.parameters(), lr=lr)
    prima = torch.sigmoid(rete(x)).item()
    for _ in range(passi):
        loss = errore(rete(x), y)
        ottimizzatore.zero_grad()
        loss.backward()
        ottimizzatore.step()
    return prima, torch.sigmoid(rete(x)).item()

for lr in [0.01, 0.1, 1]:
    prima, dopo = allena(lr)
    print(f"learning rate {lr}: da {prima:.0%} a {dopo:.1%}")

learning rate 0.01: da 45% a 35.2%
learning rate 0.1: da 45% a 9.7%
learning rate 1: da 45% a 1.0%


La soluzione nel libro dà i numeri del seme 0 e, per confronto, l'andamento su cento partenze diverse. Eccoli.

In [3]:
prima, _ = allena(0.1, seme=0)
controlla("partenza con il seme 0 (%)", prima * 100, 45, 0.5)
for lr, atteso in [(0.01, 35), (0.1, 10), (1, 1)]:
    controlla(f"seme 0, learning rate {lr} (%)", allena(lr, seme=0)[1] * 100, atteso, 0.5)

def morto(lr, seme, passi=100):
    torch.manual_seed(seme)
    rete = nn.Sequential(nn.Linear(3, 2), nn.ReLU(), nn.Linear(2, 1))
    x = torch.tensor([[0.8, 1.0, 1.0]]); y = torch.tensor([[0.0]])
    errore = nn.BCEWithLogitsLoss(); ott = torch.optim.SGD(rete.parameters(), lr=lr)
    for _ in range(passi):
        loss = errore(rete(x), y); ott.zero_grad(); loss.backward(); ott.step()
    return bool((rete[0](x) <= 0).all())   # entrambi i neuroni nascosti spenti

partenze = [allena(0.01, seme=s)[0] for s in range(100)]
print(f"partenze: dal {min(partenze):.0%} al {max(partenze):.0%}")
for lr in [0.01, 0.1, 1]:
    finali = sorted(allena(lr, seme=s)[1] for s in range(100))
    print(f"learning rate {lr}: dal {finali[0]:.1%} al {finali[-1]:.1%}, a metà {finali[50]:.1%}")
print(f"neuroni morti dopo 100 passi, con learning rate 0,1: {sum(morto(0.1, s) for s in range(100))} partenze su 100")

✓  partenza con il seme 0 (%): calcolato 44.91, nel libro 45
✓  seme 0, learning rate 0.01 (%): calcolato 35.18, nel libro 35


✓  seme 0, learning rate 0.1 (%): calcolato 9.709, nel libro 10
✓  seme 0, learning rate 1 (%): calcolato 1.002, nel libro 1


partenze: dal 24% al 74%


learning rate 0.01: dal 11.3% al 52.7%, a metà 30.2%


learning rate 0.1: dal 0.6% al 11.3%, a metà 2.9%


learning rate 1: dal 0.0% al 1.0%, a metà 0.1%


neuroni morti dopo 100 passi, con learning rate 0,1: 46 partenze su 100


## JAX: il gradiente come trasformazione

In [4]:
import jax
import jax.numpy as jnp

def errore(w, x, y):
    p = jax.nn.sigmoid(jnp.dot(w, x))
    return -(y * jnp.log(p) + (1 - y) * jnp.log(1 - p))

colpe = jax.grad(errore)
g = colpe(jnp.array([2.0, -1.5, 0.1]), jnp.array([0.8, 1.0, 1.0]), 0.0)
for nome, val, atteso in zip(["energia", "pioggia", "compagno"], g, [0.44, 0.55, 0.55]):
    controlla(f"colpa {nome}", float(val), atteso)

✓  colpa energia: calcolato 0.4399, nel libro 0.44
✓  colpa pioggia: calcolato 0.5498, nel libro 0.55
✓  colpa compagno: calcolato 0.5498, nel libro 0.55


**Esercizio 2.** Il bias come quarto peso, con un input che vale sempre 1.

In [5]:
w4 = jnp.array([2.0, -1.5, 0.1, -0.5])
x4 = jnp.array([0.8, 1.0, 1.0, 1.0])
g4 = colpe(w4, x4, 0.0)
p = float(jax.nn.sigmoid(jnp.dot(w4, x4)))
controlla("probabilità", p, 0.43)
controlla("colpa del bias", float(g4[3]), 0.43)
assert abs(float(g4[3]) - (p - 0)) < 1e-6
print("✓  la colpa del bias è uguale al segnale d'errore p − y")

✓  probabilità: calcolato 0.4256, nel libro 0.43
✓  colpa del bias: calcolato 0.4256, nel libro 0.43
✓  la colpa del bias è uguale al segnale d'errore p − y
